In [0]:
import requests
import json

# ── Connection ─────────────────────────────────────────────────────────────

host  = spark.conf.get("spark.databricks.workspaceUrl")
token = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type" : "application/json"
}

# ── Compute ────────────────────────────────────────────────────────────────

EXISTING_CLUSTER_ID = "0220-160255-4reafcow"

# ── Notebook Paths ─────────────────────────────────────────────────────────

ROOT      = "/Users/s819534@8451.com/data_sentinel_barrows"
PIPE_PATH = f"{ROOT}/agent1/pipeline"

VALIDATION_AGENT_PATH = f"{PIPE_PATH}/02_validation_agent"
AI_SUMM_PATH          = f"{PIPE_PATH}/03_ai_summarization"
ALERT_DISP_PATH       = f"{PIPE_PATH}/04_teams_alert_dispatcher"

# ── Notification Email ─────────────────────────────────────────────────────

NOTIFICATION_EMAIL = "s819534@8451.com"

print("=" * 60)
print("AGENT 1 — WORKFLOW CREATION")
print("=" * 60)
print(f"Workspace host  : {host}")
print(f"Cluster ID      : {EXISTING_CLUSTER_ID}")
print(f"Notification    : {NOTIFICATION_EMAIL}")
print()
print("Notebook paths:")
print(f"  Validation agent  : {VALIDATION_AGENT_PATH}")
print(f"  AI summarization  : {AI_SUMM_PATH}")
print(f"  Alert dispatcher  : {ALERT_DISP_PATH}")
print("=" * 60)

## Step 1: Verify Notebooks Exist Before Creating Workflow

In [0]:
def verify_notebook_exists(path: str) -> bool:
    """
    Checks whether a notebook exists at the given workspace path.
    """
    response = requests.get(
        f"https://{host}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path}
    )
    exists = response.status_code == 200
    status = "EXISTS  ✅" if exists else "MISSING ❌"
    print(f"  {status}  {path}")
    return exists

print("Verifying notebook paths...")
all_exist = all([
    verify_notebook_exists(VALIDATION_AGENT_PATH),
    verify_notebook_exists(AI_SUMM_PATH),
    verify_notebook_exists(ALERT_DISP_PATH)
])

if not all_exist:
    raise Exception(
        "One or more notebooks are missing. "
        "Please create them before running this workflow creation script."
    )

print("\nAll notebooks verified. Proceeding to create workflow.")

## Step 2: Define the Workflow

In [0]:
workflow = {
    "name": "agent1_raw_ingestion_validation",

    # ── Schedule ───────────────────────────────────────────────────────────
    # Created PAUSED so we can test manually first
    # Change pause_status to UNPAUSED for production
    "schedule": {
        "quartz_cron_expression": "0 0 6 * * ?",
        "timezone_id"           : "UTC",
        "pause_status"          : "PAUSED"
    },

    # ── Notifications ──────────────────────────────────────────────────────
    "email_notifications": {
        "on_failure"               : [NOTIFICATION_EMAIL],
        "on_success"               : [],
        "no_alert_for_skipped_runs": True
    },

    # ── Run settings ───────────────────────────────────────────────────────
    "max_concurrent_runs": 1,
    "timeout_seconds"    : 7200,

    # ── Tags ───────────────────────────────────────────────────────────────
    "tags": {
        "team"   : "data-engineering",
        "project": "data-trust-layer",
        "agent"  : "agent1"
    },

    # ── Job Parameters ─────────────────────────────────────────────────────
    # Single source of truth for target catalog/schema. Exposed to every
    # notebook task below as widgets ("catalog" / "schema"), so promoting
    # the whole job to a new catalog/schema only requires editing these
    # two defaults instead of every notebook.
    "parameters": [
        {"name": "catalog", "default": "learning_dev"},
        {"name": "schema"  , "default": "s819534"}
    ],

    # ── Tasks ─────────────────────────────────────────────────────────────
    "tasks": [

        # ── Task 1: Validation Agent ───────────────────────────────────────
        {
            "task_key"        : "run_validation_agent",
            "description"     : (
                "Runs all deterministic validation checks "
                "for the given business date across all "
                "active registered tables."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "notebook_task"   : {
                "notebook_path"   : VALIDATION_AGENT_PATH,
                "base_parameters" : {
                    "business_date": "",   # blank = yesterday
                    "table_filter" : ""    # blank = all tables
                },
                "source"          : "WORKSPACE"
            },
            "timeout_seconds"          : 3600,
            "max_retries"              : 1,
            "min_retry_interval_millis": 300000,
            "retry_on_timeout"         : False,
            "email_notifications"      : {
                "on_failure": [NOTIFICATION_EMAIL]
            }
        },

        # ── Task 2: AI Summarization ───────────────────────────────────────
        {
            "task_key"           : "run_ai_summarization",
            "description"        : (
                "Enriches PENDING validation runs with "
                "AI-generated summaries and recommended actions."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {
                    "task_key": "run_validation_agent"
                    # No outcome filter = runs after Task 1 completes
                    # regardless of whether it succeeded or failed
                    # This ensures AI runs even if agent had partial errors
                }
            ],
            "notebook_task"      : {
                "notebook_path": AI_SUMM_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 1800,
            "max_retries"              : 1,
            "min_retry_interval_millis": 120000,
            "retry_on_timeout"         : False
        },

        # ── Task 3: Teams Alert Dispatcher ────────────────────────────────
        {
            "task_key"           : "run_alert_dispatcher",
            "description"        : (
                "Sends Teams alerts for all CRITICAL and WARNING "
                "validation runs that have not yet been alerted."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {
                    "task_key": "run_ai_summarization"
                    # Runs after AI summarization regardless of outcome
                    # Deterministic results are always alertable
                    # even if AI enrichment failed
                }
            ],
            "notebook_task"      : {
                "notebook_path": ALERT_DISP_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 900,
            "max_retries"              : 2,
            "min_retry_interval_millis": 60000,
            "retry_on_timeout"         : False
        }
    ]
}

print("Workflow definition ready.")
print(f"Tasks defined : {len(workflow['tasks'])}")
for task in workflow["tasks"]:
    print(f"  {task['task_key']}")

## Step 3: Create the Workflow

In [0]:
print("Creating workflow...")

response = requests.post(
    f"https://{host}/api/2.1/jobs/create",
    headers=headers,
    json=workflow
)

print(f"Status code : {response.status_code}")
print(f"Response    : {json.dumps(response.json(), indent=2)}")

if response.status_code == 200:
    job_id = response.json().get("job_id")
    print(f"\nWorkflow created successfully.")
    print(f"Job ID  : {job_id}")
    print(f"View at : https://{host}/#job/{job_id}")
    print()
    print("Schedule is PAUSED.")
    print("Run the next cell to trigger a manual test run.")
else:
    print(f"\nWorkflow creation failed.")
    print(f"Error: {response.text}")
    job_id = None

## Step 4: Trigger a Manual Test Run

In [0]:
# Set this to the job_id from the cell above
# It is automatically set if you ran the previous cell in this session
if job_id:
    print(f"Using job_id from previous cell: {job_id}")
else:
    job_id = input("Paste job_id here: ")

# Choose a business date to test with
# Use a controlled batch date that has known quality issues
# so you can verify the full pipeline fires correctly
TEST_BUSINESS_DATE = "2026-06-24"

trigger_payload = {
    "job_id"          : int(job_id),
    "notebook_params" : {
        "business_date": TEST_BUSINESS_DATE,
        "table_filter" : ""
    }
}

print(f"\nTriggering manual test run...")
print(f"Job ID         : {job_id}")
print(f"Business date  : {TEST_BUSINESS_DATE}")

trigger_response = requests.post(
    f"https://{host}/api/2.1/jobs/run-now",
    headers=headers,
    json=trigger_payload
)

print(f"Status code : {trigger_response.status_code}")

if trigger_response.status_code == 200:
    run_id = trigger_response.json().get("run_id")
    print(f"\nWorkflow triggered successfully.")
    print(f"Run ID  : {run_id}")
    print(f"View at : https://{host}/#job/{job_id}/run/{run_id}")
else:
    print(f"\nTrigger failed: {trigger_response.text}")
    run_id = None

## Step 5: Monitor the Run

In [0]:
import time

def monitor_run(run_id: int, poll_seconds: int = 20):
    """
    Polls the workflow run until it completes.
    Prints task-level status updates.
    """
    if not run_id:
        print("No run_id available. Run the trigger cell first.")
        return

    terminal_states = {
        "TERMINATED",
        "SKIPPED",
        "INTERNAL_ERROR"
    }

    task_emojis = {
        "run_validation_agent" : "🔍",
        "run_ai_summarization" : "🤖",
        "run_alert_dispatcher" : "📣"
    }

    status_emojis = {
        "RUNNING"        : "⏳",
        "TERMINATED"     : "✅",
        "INTERNAL_ERROR" : "❌",
        "PENDING"        : "⏸",
        "SKIPPED"        : "⏭"
    }

    print(f"Monitoring run {run_id}...")
    print(f"Polling every {poll_seconds} seconds.")
    print()

    previous_task_states = {}

    while True:
        resp = requests.get(
            f"https://{host}/api/2.1/jobs/runs/get",
            headers=headers,
            params={"run_id": run_id}
        )

        if resp.status_code != 200:
            print(f"Error polling run: {resp.status_code} {resp.text}")
            break

        run_data   = resp.json()
        state      = run_data.get("state", {})
        life_state = state.get("life_cycle_state", "UNKNOWN")
        result     = state.get("result_state", "")
        tasks      = run_data.get("tasks", [])

        from datetime import datetime
        timestamp  = datetime.now().strftime("%H:%M:%S")

        print(f"[{timestamp}] Overall: {life_state} {result}")

        for task in tasks:
            task_key    = task.get("task_key", "")
            task_state  = task.get("state", {})
            task_life   = task_state.get("life_cycle_state", "PENDING")
            task_result = task_state.get("result_state", "")
            task_emoji  = task_emojis.get(task_key, "⚙️")
            state_emoji = status_emojis.get(task_life, "❓")

            current = f"{task_life} {task_result}"

            if previous_task_states.get(task_key) != current:
                previous_task_states[task_key] = current
                print(
                    f"  {task_emoji} {task_key:<30} "
                    f"{state_emoji} {task_life:<15} "
                    f"{task_result}"
                )

        print()

        if life_state in terminal_states:
            print("=" * 50)
            if result == "SUCCESS":
                print("✅ Workflow completed successfully.")
            else:
                print(f"❌ Workflow ended with: {result}")
                msg = state.get("state_message", "")
                if msg:
                    print(f"   Message: {msg}")
            print("=" * 50)
            break

        time.sleep(poll_seconds)


if run_id:
    monitor_run(int(run_id))

## Step 6: Enable the Schedule (Run After Successful Test)

In [0]:
def enable_schedule(job_id: int):
    """
    Unpauses the workflow schedule so it runs automatically
    every day at 06:00 AM UTC.
    Only run this after a successful manual test.
    """
    response = requests.patch(
        f"https://{host}/api/2.1/jobs/{job_id}",
        headers=headers,
        json={
            "new_settings": {
                "schedule": {
                    "quartz_cron_expression": "0 0 6 * * ?",
                    "timezone_id"           : "UTC",
                    "pause_status"          : "UNPAUSED"
                }
            }
        }
    )

    print(f"Status code : {response.status_code}")

    if response.status_code == 200:
        print(f"Schedule enabled successfully.")
        print(f"Workflow will now run daily at 06:00 AM UTC.")
        print(f"View at: https://{host}/#job/{job_id}")
    else:
        print(f"Failed to enable schedule: {response.text}")


# Uncomment and run this line after successful test
# enable_schedule(int(job_id))
print("Schedule is currently PAUSED.")
print("Uncomment enable_schedule() and rerun this cell after a successful test.")

## Step 7: Verify End-to-End Results

In [0]:
print("Verifying end-to-end results after the workflow run...")

spark.sql(f"""
    SELECT
        business_date,
        SPLIT(fully_qualified_name, '\\.')[2] AS table_name,
        overall_status,
        total_checks_run,
        total_checks_passed,
        total_checks_warned,
        total_checks_critical,
        ai_enrichment_status,
        CASE WHEN alert_sent = true THEN 'YES' ELSE 'NO' END AS alert_fired,
        alert_sent_timestamp
    FROM learning_dev.s819534.uc1_validation_run_b
    WHERE business_date = '{TEST_BUSINESS_DATE}'
    ORDER BY fully_qualified_name
""").show(truncate=False)